In [6]:
import re
import math
import os
import pandas as pd

# =============================================================================
# 1. Data Parser
# =============================================================================
def parse_dat_file(dat_filepath):
    # Defaults matching the latest logic
    params = {
        'scaleLength': 635.0,
        'stringSpacing': 10.8,
        'timePerMm': 0.25,
        's_max': 6, 'f_max': 24, 'p_max': 24,
        'fingerLengths_mm': [70.0, 75.0, 70.0, 50.0],
        'fingerDifficulty': [1.0, 1.5, 2.0, 3.0],
        'natural_min_spread_mm': [20.0, 40.0, 60.0],
        'availableTime': [],
        # Weights
        'd_PC': 1.5, 'd_SC': 1.0, 'd_ST': 1.0, 'd_DH': 1.0, 'd_NO': 10.0,
        'alpha_PC': 0.2, 'alpha_SC': 0.2, 'alpha_ST': 0.2, 'alpha_DH': 0.2, 'alpha_NO': 0.2,
        'd_AC': 1.0, 'alpha_AC': 0.2
    }

    print(f"DEBUG: Parsing .dat file: {dat_filepath}")
    if not os.path.exists(dat_filepath):
        print("WARNING: File not found. Using defaults.")
        return params

    try:
        content = ""
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(dat_filepath, 'r', encoding=enc) as f:
                    content = f.read()
                break
            except UnicodeDecodeError: continue

        def get_val(name, cast_func):
            pat = re.compile(rf'{name}\s*=\s*([-+]?[0-9]*\.?[0-9]+);?')
            m = pat.search(content)
            if m: return cast_func(m.group(1))
            return None

        def get_arr(name):
            pat = re.compile(rf'{name}\s*=\s*\[(.*?)\];?', re.DOTALL)
            m = pat.search(content)
            if m:
                txt = re.sub(r'//.*', '', m.group(1)).replace('\n', ' ')
                return [float(v) for v in txt.split(',') if v.strip()]
            return None

        for k in params.keys():
            if isinstance(params[k], list):
                val = get_arr(k)
                if val: params[k] = val
            else:
                val = get_val(k, float)
                if val is not None: params[k] = val
        
        params['s_max'] = int(params['s_max'])
        params['f_max'] = int(params['f_max'])
        
        # --- Note: No auto-shift logic needed as dat generation is fixed ---
        
        return params
    except Exception as e:
        print(f"ERROR: Parsing failed ({e}). Using defaults.")
        return params

def parse_solution_file(solution_filepath):
    notes_data = []
    if not os.path.exists(solution_filepath): return []
    
    try:
        lines = []
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(solution_filepath, 'r', encoding=enc) as f:
                    lines = f.readlines()
                break
            except UnicodeDecodeError: continue

        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue
            if not header_found:
                if "Note" in line and "String" in line: header_found = True
                continue
            if line.startswith('---'): break
            
            parts = re.split(r'\s+', line)
            if parts[0] == "Note": continue
            if len(parts) >= 5:
                try:
                    # Handle Rest Note 'R'
                    if parts[0] == 'R':
                         note = {
                            'note': 'R', 'string': 0, 'fret': 0, 'hand': 0, 'finger': 0
                        }
                    else:
                        note = {
                            'note': parts[0],
                            'string': int(parts[1]),
                            'fret': int(parts[2]),
                            'hand': int(parts[3]),
                            'finger': 0 if parts[4] == 'x' else int(parts[4])
                        }
                    notes_data.append(note)
                except ValueError: continue
        return notes_data
    except: return []

# =============================================================================
# 2. Updated Model Logic (Synced with OPL & STcheck)
# =============================================================================
class GuitarModelReproducer:
    def __init__(self, params):
        self.p = params
        self.fingerLengths = [0.0] + self.p['fingerLengths_mm']
        self.fingerDiff = self.p['fingerDifficulty']
        self.knuckleDist = [0.0, 0.0] + self.p['natural_min_spread_mm']
        self.limitRatios = [0.0, 1.0, 1.05, 1.05, 1.2]
        self.baseFactors = [0.0, 0.7, 0.9, 0.85, 1.0]
        self.distThresholds = [0.0, 0.6, 0.6, 0.45, 0.6]
        self.stringTension = {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.2, 5: 1.4, 6: 1.6}
        self.deg_to_rad = math.pi / 180.0
        self.rad_to_deg = 180.0 / math.pi
        self.breakAngleDeg = 15.0
        self.tanTheta = math.tan(self.breakAngleDeg * self.deg_to_rad)
        sl = self.p['scaleLength']
        self.fret_pos = [sl - (sl / (2 ** (f / 12.0))) for f in range(self.p['f_max'] + 1)]
        self.tp_base = 0.15
        self.tp_ref = 60.0
        self.tp_exp = 1.5

    def calculate_wrist_angle(self, hand_pos, finger_idx, string, fret):
        best_angle = 0.0
        min_cost = float('inf')
        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0
        for ang in range(-90, 2, 2):
            thetaRad = ang * self.deg_to_rad
            d = self.knuckleDist[finger_idx] if finger_idx < len(self.knuckleDist) else 0.0
            kX = pivotX + (d * math.cos(thetaRad))
            kY = pivotY + (d * math.sin(thetaRad))
            tX = self.fret_pos[fret]
            tY = (string - 1) * self.p['stringSpacing']
            vX = tX - kX
            vY = tY - kY
            dist = math.sqrt(vX*vX + vY*vY)
            effLen = self.fingerLengths[finger_idx] * self.baseFactors[finger_idx] if finger_idx < len(self.baseFactors) else 0.0
            if effLen > 0 and dist > effLen * self.limitRatios[finger_idx]:
                continue
            cost = dist
            if cost < min_cost:
                min_cost = cost
                best_angle = ang
        return best_angle

    def calculate_costs(self, hand_pos, finger_idx, string, fret):
        if fret < hand_pos: return (100000.0, 0.0)
        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0 
        maxRotation = 40.0
        baseTheta = maxRotation * (string - 6.0) / 5.0
        fretRotationMax = (fret / 17.0) * 15.0
        stringFactor = (string - 1.0) / 5.0
        fretRotation = fretRotationMax * stringFactor
        thetaDeg = baseTheta - fretRotation
        thetaRad = thetaDeg * self.deg_to_rad
        d = self.knuckleDist[finger_idx]
        kX = pivotX + (d * math.cos(thetaRad))
        kY = pivotY + (d * math.sin(thetaRad))
        tX = self.fret_pos[fret]
        tY = (string - 1) * self.p['stringSpacing']
        vX = tX - kX; vY = tY - kY
        dist = math.sqrt(vX*vX + vY*vY)
        dotProd = vX * -math.sin(thetaRad) + vY * math.cos(thetaRad)
        cosDelta = dotProd / dist if dist > 0.001 else 1.0
        cosDelta = max(-1.0, min(1.0, cosDelta))
        deltaDeg = math.acos(cosDelta) * self.rad_to_deg
        neckFactor = 1.0
        if fret >= 15:
            ratioHigh = (fret - 15.0) / (self.p['f_max'] - 15.0)
            if ratioHigh > 1.0: ratioHigh = 1.0
            neckFactor = 1.0 - (0.15 * ratioHigh)
        eff_len = self.fingerLengths[finger_idx] * self.baseFactors[finger_idx] * neckFactor
        limit = self.limitRatios[finger_idx]
        if eff_len > 0 and dist > eff_len * limit:
            return (100000.0, 0.0)
        distCost = 0.0
        ratio = 0.0
        threshold = self.distThresholds[finger_idx]
        if eff_len > 0:
            ratio = dist / eff_len
            if ratio < 0.3: distCost = (0.3 - ratio) * 5.0
            elif ratio <= threshold: distCost = 0.0
            else:
                distCost = (ratio - threshold) * 10.0
                if ratio > 1.0: distCost += (ratio - 1.0) * 20.0
        baseMaxAngle = 20.0 if finger_idx == 4 else 15.0
        maxAngle = baseMaxAngle
        if fret >= 12:
            maxAngle = max(10.0, baseMaxAngle - (fret - 12) * 0.5)
        angleMultiplier = 0.5
        if eff_len > 0 and 0.3 <= ratio <= threshold:
            angleMultiplier = 0.2
        angleCost = (deltaDeg - maxAngle) * angleMultiplier if deltaDeg > maxAngle else 0.0
        diff_val = self.fingerDiff[finger_idx - 1]
        st_val = (distCost + angleCost) * diff_val
        a_mm = 0.5 + (2.0 * fret / 12.0)
        l_mm = self.p['scaleLength'] - self.fret_pos[fret]
        tau = self.stringTension.get(string, 1.0)
        if l_mm <= 0: forceIndex = 0
        else: forceIndex = (2.0 * tau * a_mm) / (l_mm * self.tanTheta)
        ph_val = (forceIndex * 100.0) * diff_val
        return (st_val, ph_val)

    def calculate_total(self, notes_data):
        total = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0, 'AC': 0.0}
        details = []
        num_notes = len(notes_data)
        avail_time = self.p['availableTime']
        if len(avail_time) < num_notes:
            avail_time += [30.0] * (num_notes - len(avail_time))
        prev_idx_map = {}
        acc_time = {}
        wrist_angles = []

        for i in range(num_notes):
            is_rest = (notes_data[i]['note'] == 'R')
            if not is_rest:
                p = i - 1
                t_sum = 0.0
                while p >= 0 and notes_data[p]['note'] == 'R':
                    t_sum += avail_time[p]
                    p -= 1
                if p >= 0:
                    prev_idx_map[i] = p
                    acc_time[i] = t_sum + avail_time[p]
                else:
                    prev_idx_map[i] = i
                    acc_time[i] = 0.0
            else:
                acc_time[i] = 0.0
                prev_idx_map[i] = -1

        for i in range(num_notes):
            curr = notes_data[i]
            if curr['note'] == 'R':
                wrist_angles.append(0.0)
            else:
                finger_idx = curr['finger'] if isinstance(curr['finger'], int) and curr['finger'] in [1,2,3,4] else 1
                hand_pos = curr['hand'] if 'hand' in curr else 1
                wrist_angles.append(self.calculate_wrist_angle(hand_pos, finger_idx, curr['string'], curr['fret']))

        for i in range(num_notes):
            curr = notes_data[i]
            row = {'PC': 0.0, 'SC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0, 'AC': 0.0}
            time_penalty = 0.0
            if curr['note'] != 'R':
                prev_i = prev_idx_map.get(i, i)
                if prev_i == i:
                    time_penalty = 0.0
                else:
                    t_val = acc_time[i] + 0.01
                    time_penalty = self.tp_base + (self.tp_ref / t_val) ** self.tp_exp
                if curr['fret'] == 0:
                    row['NO'] = 1.0
                if curr['finger'] in [1,2,3,4] and curr['fret'] > 0:
                    st_raw, ph_raw = self.calculate_costs(curr['hand'], curr['finger'], curr['string'], curr['fret'])
                    if st_raw >= 50000: st_raw = 100000.0
                    row['ST'] = st_raw
                    row['DH'] = ph_raw
                if prev_i != i and prev_i != -1:
                    prev = notes_data[prev_i]
                    row['PC'] = abs(self.fret_pos[prev['hand']] - self.fret_pos[curr['hand']])
                    row['SC'] = self.p['stringSpacing'] * abs(prev['string'] - curr['string'])
                    if (curr['finger'] == 1 and prev['finger'] == 1 and curr['fret'] == prev['fret'] and abs(curr['string'] - prev['string']) >= 1):
                        row['SC'] -= (self.p['stringSpacing'] * 0.9)
                    ac_val = abs(wrist_angles[i] - wrist_angles[prev_i])
                    row['AC'] = ac_val * self.p['d_AC'] * self.p['alpha_AC']
            w_pc = row['PC'] * time_penalty * self.p['d_PC'] * self.p['alpha_PC']
            w_sc = row['SC'] * time_penalty * self.p['d_SC'] * self.p['alpha_SC']
            w_st = row['ST'] * self.p['d_ST'] * self.p['alpha_ST'] 
            w_dh = row['DH'] * self.p['d_DH'] * self.p['alpha_DH']
            w_no = row['NO'] * self.p['d_NO'] * self.p['alpha_NO']
            w_ac = row['AC']
            total['PC'] += w_pc
            total['SC'] += w_sc
            total['ST'] += w_st
            total['DH'] += w_dh
            total['NO'] += w_no
            total['AC'] += w_ac
            sum_weighted = w_pc + w_sc + w_st + w_dh + w_no + w_ac
            details.append({
                'Index': i+1, 'Note': curr['note'],
                'Time(ms)': acc_time.get(i, 0),
                'TimeFactor': time_penalty,
                'PC_w': w_pc, 'SC_w': w_sc, 'ST_w': w_st, 'PH_w': w_dh,
                'NO_w': w_no, 'AC_w': w_ac,
                'ST_raw': row['ST'], 'PH_raw': row['DH'], 'AC_raw': row['AC'],
                'WristAngle': wrist_angles[i],
                'Total': sum_weighted
            })
        grand_total = sum(total.values())
        return total, grand_total, pd.DataFrame(details)

In [7]:
# --- 実行・表示部分 ---
if __name__ == "__main__":
    # --- Changed filename to complex_solo_mod.dat ---
    dat_file = "datas/complex_solo_mod.dat"
    sol_file = "results/complex_solo_mod_modified_model (39).txt"
    
    if os.path.exists(dat_file) and os.path.exists(sol_file):
        params = parse_dat_file(dat_file)
        model = GuitarModelReproducer(params)
        notes = parse_solution_file(sol_file)
        
        if notes:
            bd, total, df = model.calculate_total(notes)
            print("\n" + "="*60)
            print(" --- Updated Cost Breakdown (With Diff Multiplier) ---")
            print(f"  Position Change (PC): {bd['PC']:.4f}")
            print(f"  String Change   (SC): {bd['SC']:.4f}")
            print(f"  Geo Stretch     (ST): {bd['ST']:.4f}")
            print(f"  Phys Action     (DH): {bd['DH']:.4f}")
            print(f"  Open String     (NO): {bd['NO']:.4f}")
            print(f"  Angle Change    (AC): {bd['AC']:.4f}")
            print("-" * 40)
            print(f"  Total Cost (Sum)    : {total:.4f}")
            print("="*60)
            if not df.empty:
                print("\n[Details - All Notes]")
                pd.set_option('display.max_columns', None)
                pd.set_option('display.width', 1000)
                pd.set_option('display.max_rows', None)
                print(df[['Index', 'Note', 'Time(ms)', 'TimeFactor', 'PC_w', 'SC_w', 'ST_w', 'PH_w', 'NO_w', 'AC_w', 'ST_raw', 'PH_raw', 'AC_raw', 'WristAngle', 'Total']].to_string(index=False, float_format="%.3f"))
    else:
        print("Please upload 'datas/complex_solo_mod.dat' and 'complex_opt.txt' to run the calculation.")

DEBUG: Parsing .dat file: datas/complex_solo_mod.dat

 --- Updated Cost Breakdown (With Diff Multiplier) ---
  Position Change (PC): 44.9417
  String Change   (SC): 53.3091
  Geo Stretch     (ST): 49.2151
  Phys Action     (DH): 142.8260
  Open String     (NO): 0.0000
  Angle Change    (AC): 162.0000
----------------------------------------
  Total Cost (Sum)    : 452.2919

[Details - All Notes]
 Index Note  Time(ms)  TimeFactor  PC_w  SC_w  ST_w  PH_w  NO_w  AC_w  ST_raw  PH_raw  AC_raw  WristAngle  Total
     1   A4     0.000       0.000 0.000 0.000 0.000 1.256 0.000 0.000   0.000   8.972   0.000       0.000  1.256
     2   A4   189.000       0.329 0.000 0.000 0.000 1.256 0.000 0.000   0.000   8.972   0.000       0.000  1.256
     3   A4   190.000       0.327 0.000 0.000 0.000 1.256 0.000 0.000   0.000   8.972   0.000       0.000  1.256
     4   G4    95.000       0.652 0.000 0.000 0.794 0.823 0.000 3.600   2.647   5.877   3.600     -90.000  5.217
     5   F4    95.000       0.652 0.